Vector store-backed retriever

这是最基本的检索实现方式，用向量库的实现方法，例如相似性搜索和MMR搜索向量库中的文本。使用时仅需要加载embedding模型。

支持按topK、相似度得分门限以及MMR等多种方式检索相关文档。

```
pip install -U langchain
pip install langchain-text-splitters
pip install pypdf
pip install chromadb
pip install dashscope
pip install ipywidgets
```

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma

import sys
import os
sys.path.append("./")
from rag.rag_lecture_materials.config.config import RagConfig

/tmp/ipykernel_2259930/319038007.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
# 向量模型
embeddings = OpenAIEmbeddings(model="Pro/BAAI/bge-m3", base_url="https://api.siliconflow.cn/v1", api_key=RagConfig.api_key)
print(embeddings)

client=<openai.resources.embeddings.Embeddings object at 0x727b6882c830> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x727b6882d7f0> model='Pro/BAAI/bge-m3' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base='https://api.siliconflow.cn/v1' openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True


In [ ]:
loader = PyPDFLoader("../data/西游记影评.pdf")
# 加载分割文档
documents = loader.load_and_split()
#指定chunk大小
text_splitter = RecursiveCharacterTextSplitter(separators=["。"], chunk_size=128, chunk_overlap=10)
texts_chunks = text_splitter.split_documents(documents)

In [4]:
documents

[Document(metadata={'producer': '', 'creator': 'WPS 文字', 'creationdate': '2025-05-19T20:36:10+08:00', 'author': '航', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2025-05-19T20:36:10+08:00', 'sourcemodified': "D:20250519203610+08'00'", 'subject': '', 'title': '', 'trapped': '/False', 'source': './西游记影评.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='在众多影视作品中，《西游记》以其独特魅力脱颖而出，成为难以逾越的经典。\n这部改编自明代吴承恩同名巨著的作品，历经岁月洗礼，愈发熠熠生辉，蕴含的\n丰富文化内涵和深刻哲理，深深影响着一代又一代观众。\n《西游记》的剧情围绕唐僧师徒四人西天取经展开，他们一路上降妖除魔、历经\n九九八十一难，最终取得真经、修成正果。看似简单的故事框架，实则包含无数\n精彩情节与深刻寓意。从石猴出世、大闹天宫的叛逆反抗，到踏上取经之路后历\n经磨难的成长蜕变，剧情跌宕起伏、扣人心弦。像三打白骨精、女儿国奇遇、真\n假美猴王等经典情节，不仅展现了师徒四人面临的艰难险阻，更通过对人性、道\n德、信仰的考验，引发观众的深入思考。比如，白骨精三次变幻人形，迷惑唐僧\n师徒，孙悟空火眼金睛识破并将其打死，却因唐僧误解而被逐出师门。这一情节\n深刻揭示了人性的复杂与善恶难辨，也凸显了坚持真理的艰难。\n剧中人物形象鲜明饱满，令人印象深刻。孙悟空是核心人物，他机智勇敢、神通\n广大，拥有七十二变和火眼金睛，一个筋斗云便能翻出十万八千里。他的金箍棒\n威力无穷，打得妖魔鬼怪闻风丧胆。同时，他又具有强烈的叛逆精神，敢于挑战\n天庭权威，喊出 “皇帝轮流做，明年到我家” 的豪言壮语。但在取经过程中，他\n逐渐学会克制与担当，从最初的顽劣猴王成长为守护正义的斗战胜佛，其成长历\n程激励着无数观众勇敢追求梦想、战胜困难。唐僧慈悲善良、信念坚定，一心向\n

In [5]:
texts_chunks

[Document(metadata={'producer': '', 'creator': 'WPS 文字', 'creationdate': '2025-05-19T20:36:10+08:00', 'author': '航', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2025-05-19T20:36:10+08:00', 'sourcemodified': "D:20250519203610+08'00'", 'subject': '', 'title': '', 'trapped': '/False', 'source': './西游记影评.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='在众多影视作品中，《西游记》以其独特魅力脱颖而出，成为难以逾越的经典。\n这部改编自明代吴承恩同名巨著的作品，历经岁月洗礼，愈发熠熠生辉，蕴含的\n丰富文化内涵和深刻哲理，深深影响着一代又一代观众'),
 Document(metadata={'producer': '', 'creator': 'WPS 文字', 'creationdate': '2025-05-19T20:36:10+08:00', 'author': '航', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2025-05-19T20:36:10+08:00', 'sourcemodified': "D:20250519203610+08'00'", 'subject': '', 'title': '', 'trapped': '/False', 'source': './西游记影评.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='。\n《西游记》的剧情围绕唐僧师徒四人西天取经展开，他们一路上降妖除魔、历经\n九九八十一难，最终取得真经、修成正果。看似简单的故事框架，实则包含无数\n精彩情节与深刻寓意'),
 Document(metadata={'producer': '', 

In [6]:
# 存入向量库，创建retriever,将当前目录下创建db为持久化向量数据库的文件,如果已经存在则自动加载
vectorstore = Chroma.from_documents(texts_chunks, embeddings,collection_name="novel", persist_directory="db")
# 然后你就可以像平常一样使用vectorstore了
query="西游记"
retriever2 = vectorstore.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.1}
)
docs = retriever2.invoke(query)
docs

[Document(metadata={'author': '航', 'creator': 'WPS 文字', 'comments': '', 'source': './西游记影评.pdf', 'sourcemodified': "D:20250519203610+08'00'", 'page_label': '2', 'page': 1, 'subject': '', 'trapped': '/False', 'title': '', 'keywords': '', 'producer': '', 'company': '', 'total_pages': 2, 'moddate': '2025-05-19T20:36:10+08:00', 'creationdate': '2025-05-19T20:36:10+08:00'}, page_content='。以《西游记》为\n蓝本的各类衍生作品层出不穷，如电影、动画、游戏、舞台剧等，不断丰富着 “西\n游” 文化的内涵，使其在新时代焕发出新的活力'),
 Document(metadata={'source': './西游记影评.pdf', 'sourcemodified': "D:20250519203610+08'00'", 'moddate': '2025-05-19T20:36:10+08:00', 'company': '', 'comments': '', 'creationdate': '2025-05-19T20:36:10+08:00', 'total_pages': 2, 'title': '', 'page_label': '1', 'creator': 'WPS 文字', 'producer': '', 'page': 0, 'subject': '', 'author': '航', 'trapped': '/False', 'keywords': ''}, page_content='。师徒四人性格迥异，却相互配合、互补长\n短，共同构成一个紧密团结的团队，完美诠释了团队合作的重要性。\n《西游记》蕴含的文化内涵博大精深。它融合了佛、道、儒三家思想，展现出中\n国传统文化的独特魅力'),
 Document(metadata={'subject': '', 'keywords': ''

In [7]:
# ----------------- (1) 指定topK检索 -------------------------- #
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
docs = retriever.invoke(query)
print("TOPK检索结果:",[k.page_content for k in docs])


TOPK检索结果: ['。以《西游记》为\n蓝本的各类衍生作品层出不穷，如电影、动画、游戏、舞台剧等，不断丰富着 “西\n游” 文化的内涵，使其在新时代焕发出新的活力', '。师徒四人性格迥异，却相互配合、互补长\n短，共同构成一个紧密团结的团队，完美诠释了团队合作的重要性。\n《西游记》蕴含的文化内涵博大精深。它融合了佛、道、儒三家思想，展现出中\n国传统文化的独特魅力']


In [8]:
# ----------------- (1) 指定topK检索另一种方式 -------------------------- #
result = vectorstore.similarity_search("你好", 4)
for hits in result:
    for hit in hits:
        print(hit)

('id', None)
('metadata', {'source': './西游记影评.pdf', 'subject': '', 'title': '', 'moddate': '2025-05-19T20:36:10+08:00', 'page_label': '1', 'creationdate': '2025-05-19T20:36:10+08:00', 'trapped': '/False', 'producer': '', 'author': '航', 'creator': 'WPS 文字', 'total_pages': 2, 'keywords': '', 'page': 0, 'sourcemodified': "D:20250519203610+08'00'", 'comments': '', 'company': ''})
('page_content', '。比如，白骨精三次变幻人形，迷惑唐僧\n师徒，孙悟空火眼金睛识破并将其打死，却因唐僧误解而被逐出师门。这一情节\n深刻揭示了人性的复杂与善恶难辨，也凸显了坚持真理的艰难。\n剧中人物形象鲜明饱满，令人印象深刻')
('type', 'Document')
('id', None)
('metadata', {'page_label': '2', 'subject': '', 'source': './西游记影评.pdf', 'creator': 'WPS 文字', 'moddate': '2025-05-19T20:36:10+08:00', 'keywords': '', 'total_pages': 2, 'comments': '', 'producer': '', 'author': '航', 'sourcemodified': "D:20250519203610+08'00'", 'trapped': '/False', 'title': '', 'company': '', 'page': 1, 'creationdate': '2025-05-19T20:36:10+08:00'})
('page_content', '。云雾缭\n绕的花果山、金碧辉煌的天宫、阴森恐怖的妖洞，每一处场景都充满想象力，令\n观众身临其境。在人物造型设计上，孙悟空的虎皮裙、金箍，猪八戒的大耳朵、

In [10]:
# -----------------(2) 根据相似度门限值检索 ----------------- #
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.1} #l2 distance is default
)
docs = retriever.invoke(query)
print("相似度阈值检索结果:",[k.page_content for k in docs])

相似度阈值检索结果: ['。以《西游记》为\n蓝本的各类衍生作品层出不穷，如电影、动画、游戏、舞台剧等，不断丰富着 “西\n游” 文化的内涵，使其在新时代焕发出新的活力', '。师徒四人性格迥异，却相互配合、互补长\n短，共同构成一个紧密团结的团队，完美诠释了团队合作的重要性。\n《西游记》蕴含的文化内涵博大精深。它融合了佛、道、儒三家思想，展现出中\n国传统文化的独特魅力', '。此外，剧中对古代民俗文化、神话传说的描绘', '在众多影视作品中，《西游记》以其独特魅力脱颖而出，成为难以逾越的经典。\n这部改编自明代吴承恩同名巨著的作品，历经岁月洗礼，愈发熠熠生辉，蕴含的\n丰富文化内涵和深刻哲理，深深影响着一代又一代观众']


In [11]:
# 根据相似度门限值检索另一种检索方式
result = vectorstore.similarity_search_with_relevance_scores("西游记", k=2,score_threshold=0.1)
for hits in result:
    for hit in hits:
        print(hit)

page_content='。以《西游记》为
蓝本的各类衍生作品层出不穷，如电影、动画、游戏、舞台剧等，不断丰富着 “西
游” 文化的内涵，使其在新时代焕发出新的活力' metadata={'keywords': '', 'comments': '', 'producer': '', 'company': '', 'source': './西游记影评.pdf', 'trapped': '/False', 'moddate': '2025-05-19T20:36:10+08:00', 'page': 1, 'sourcemodified': "D:20250519203610+08'00'", 'title': '', 'creationdate': '2025-05-19T20:36:10+08:00', 'subject': '', 'total_pages': 2, 'author': '航', 'creator': 'WPS 文字', 'page_label': '2'}
0.5848007743133582
page_content='。师徒四人性格迥异，却相互配合、互补长
短，共同构成一个紧密团结的团队，完美诠释了团队合作的重要性。
《西游记》蕴含的文化内涵博大精深。它融合了佛、道、儒三家思想，展现出中
国传统文化的独特魅力' metadata={'comments': '', 'author': '航', 'source': './西游记影评.pdf', 'subject': '', 'creationdate': '2025-05-19T20:36:10+08:00', 'total_pages': 2, 'keywords': '', 'moddate': '2025-05-19T20:36:10+08:00', 'creator': 'WPS 文字', 'producer': '', 'page_label': '1', 'sourcemodified': "D:20250519203610+08'00'", 'title': '', 'company': '', 'trapped': '/False', 'page': 0}
0.5010379589954133


In [13]:
# ----------------- (3) MMR检索:在一堆候选内容里，优先选那些既跟用户查询相关、又跟已经选过的内容不重复的条目。-------------------------- #
retriever = vectorstore.as_retriever(search_type="mmr")
docs = retriever.invoke(query)
docs


[Document(metadata={'title': '', 'company': '', 'author': '航', 'page_label': '2', 'creationdate': '2025-05-19T20:36:10+08:00', 'keywords': '', 'page': 1, 'source': './西游记影评.pdf', 'comments': '', 'producer': '', 'sourcemodified': "D:20250519203610+08'00'", 'subject': '', 'creator': 'WPS 文字', 'trapped': '/False', 'moddate': '2025-05-19T20:36:10+08:00', 'total_pages': 2}, page_content='。以《西游记》为\n蓝本的各类衍生作品层出不穷，如电影、动画、游戏、舞台剧等，不断丰富着 “西\n游” 文化的内涵，使其在新时代焕发出新的活力'),
 Document(metadata={'title': '', 'sourcemodified': "D:20250519203610+08'00'", 'company': '', 'source': './西游记影评.pdf', 'moddate': '2025-05-19T20:36:10+08:00', 'keywords': '', 'subject': '', 'comments': '', 'creationdate': '2025-05-19T20:36:10+08:00', 'page_label': '1', 'author': '航', 'trapped': '/False', 'total_pages': 2, 'producer': '', 'creator': 'WPS 文字', 'page': 0}, page_content='。师徒四人性格迥异，却相互配合、互补长\n短，共同构成一个紧密团结的团队，完美诠释了团队合作的重要性。\n《西游记》蕴含的文化内涵博大精深。它融合了佛、道、儒三家思想，展现出中\n国传统文化的独特魅力'),
 Document(metadata={'creator': 'WPS 文字', 'sourcem

In [14]:
#-------------------(4) 父子文档检索器:创建文档时包含切割后的子文档,同时保留子文档和父亲文档的关系,先匹配子文档,然后通过父子关系链接到父文档,以此获取更广阔的上下文空间-----
import uuid
from langchain_classic.retrievers import MultiVectorRetriever
from langchain_core.documents import Document
from langchain_classic.storage import InMemoryStore

In [ ]:
question_list=[]
answer_list=[]
with open("../data/金融QA数据集.csv",encoding='utf-8') as f:
    lines=f.readlines()
    print(lines)
    for i,l in enumerate(lines):
        if i==0:
            continue
        temp_question=l.split(",")[0]
        temp_answer=l.split(",")[1]
        question_list.append(temp_question)
        answer_list.append(temp_answer)

['Question,Answer\n', '解释Black-Scholes期权定价模型的核心假设和局限性。,核心假设：市场无摩擦（无交易成本、税收）；标的资产价格服从几何布朗运动（对数正态分布）；无风险利率和波动率恒定；允许无限卖空和连续交易。局限性：假设条件过于理想化，实际市场中存在交易成本和税收；标的资产价格可能不完全符合几何布朗运动；波动率并非恒定，而是随时间变化；市场并非完全无摩擦，且不允许无限卖空。\n', '什么是金融衍生品？,金融衍生品是一种金融工具，其价值取决于基础资产（如股票、债券、利率、货币等）的价格变动。常见的金融衍生品包括期货、期权、互换和远期合约。\n', '解释什么是利率互换。,利率互换是一种金融合约，交易双方同意在未来一定期限内交换现金流，通常是固定利率和浮动利率之间的交换。它用于管理利率风险，降低融资成本。\n', '什么是市盈率（P/ERatio）？,市盈率是股票价格与每股收益的比率，用于衡量股票的估值水平。它反映了投资者为每单位收益支付的价格，是评估股票投资价值的重要指标。\n', '解释什么是量化宽松（QE）。,量化宽松是央行通过购买长期债券等资产向市场注入大量资金的货币政策。其目的是降低长期利率，刺激经济增长，增加货币供应量。\n', '什么是信用评级？,信用评级是评估借款人信用风险的评级体系，通常由专业评级机构（如穆迪、标普、惠誉）给出。它反映了借款人按时偿还债务的能力和意愿。\n', '解释什么是股息率。,股息率是公司年度股息与股票价格的比率，用于衡量股票的分红收益水平。它反映了投资者通过持有股票获得现金回报的效率。\n', '什么是外汇储备？,外汇储备是一个国家持有的外币资产和黄金储备，用于平衡国际收支、稳定本国货币汇率和应对经济危机。\n', '解释什么是杠杆率。,杠杆率是指企业或投资者通过债务融资扩大资产规模的比例。高杠杆率可以放大收益，但也会增加风险。\n', '什么是通货膨胀？,通货膨胀是指货币供应量增加导致物价普遍上升的现象。它反映了货币购买力的下降。\n', '解释什么是有效市场假说。,有效市场假说认为，所有可用信息都已反映在资产价格中，资产价格总是处于合理水平。这意味着投资者难以通过分析公开信息获得超额收益。\n', '什么是资产证券化？,资产证券化是将缺乏流动性但有未来现金流的资产打包成证券出售的过程。

In [18]:
question_list

['解释Black-Scholes期权定价模型的核心假设和局限性。',
 '什么是金融衍生品？',
 '解释什么是利率互换。',
 '什么是市盈率（P/ERatio）？',
 '解释什么是量化宽松（QE）。',
 '什么是信用评级？',
 '解释什么是股息率。',
 '什么是外汇储备？',
 '解释什么是杠杆率。',
 '什么是通货膨胀？',
 '解释什么是有效市场假说。',
 '什么是资产证券化？',
 '解释什么是套期保值。',
 '什么是流动性风险？',
 '解释什么是资本充足率。',
 '什么是货币供应量？',
 '解释什么是信用风险。',
 '什么是股息再投资计划（DRIP）？',
 '解释什么是市场深度。',
 '什么是零息债券？',
 '解释什么是风险平价策略。',
 '什么是债务重组？',
 '解释什么是货币市场。',
 '什么是资本利得？',
 '解释什么是利率敏感性缺口。',
 '什么是信用利差？',
 '解释什么是量化交易。',
 '什么是外汇远期合约？',
 '解释什么是市场风险。',
 '什么是回购协议？',
 '解释什么是市净率（P/BRatio）。',
 '什么是信用违约互换（CDS）？',
 '解释什么是货币乘数。',
 '什么是资产配置？',
 '解释什么是流动性溢价。',
 '什么是利率敏感性资产？',
 '解释什么是风险价值（VaR）。',
 '什么是优先股？',
 '解释什么是久期。',
 '什么是债务与权益比率？',
 '解释什么是现金流量折现法（DCF）。',
 '什么是股票回购？',
 '解释什么是市场效率。',
 '什么是固定收益证券？',
 '解释什么是资本成本。',
 '什么是外汇期权？',
 '解释什么是系统性风险。',
 '什么是私募股权基金？',
 '解释什么是市场分割理论。',
 '什么是股票分割？',
 '解释什么是行为金融学。',
 '什么是利率互换基差？',
 '解释什么是信用利差溢价。',
 '什么是货币市场基金？',
 '解释什么是市场深度指标。',
 '什么是短期利率期货？',
 '解释什么是风险调整收益。',
 '什么是股票指数期货？',
 '解释什么是市场冲击成本。',
 '什么是长期资本管理公司（LTCM）事件？',
 '解释什么是信用风险缓释工具。',
 '什么是高收益债券？',
 '解

In [19]:
answer_list

['核心假设：市场无摩擦（无交易成本、税收）；标的资产价格服从几何布朗运动（对数正态分布）；无风险利率和波动率恒定；允许无限卖空和连续交易。局限性：假设条件过于理想化，实际市场中存在交易成本和税收；标的资产价格可能不完全符合几何布朗运动；波动率并非恒定，而是随时间变化；市场并非完全无摩擦，且不允许无限卖空。\n',
 '金融衍生品是一种金融工具，其价值取决于基础资产（如股票、债券、利率、货币等）的价格变动。常见的金融衍生品包括期货、期权、互换和远期合约。\n',
 '利率互换是一种金融合约，交易双方同意在未来一定期限内交换现金流，通常是固定利率和浮动利率之间的交换。它用于管理利率风险，降低融资成本。\n',
 '市盈率是股票价格与每股收益的比率，用于衡量股票的估值水平。它反映了投资者为每单位收益支付的价格，是评估股票投资价值的重要指标。\n',
 '量化宽松是央行通过购买长期债券等资产向市场注入大量资金的货币政策。其目的是降低长期利率，刺激经济增长，增加货币供应量。\n',
 '信用评级是评估借款人信用风险的评级体系，通常由专业评级机构（如穆迪、标普、惠誉）给出。它反映了借款人按时偿还债务的能力和意愿。\n',
 '股息率是公司年度股息与股票价格的比率，用于衡量股票的分红收益水平。它反映了投资者通过持有股票获得现金回报的效率。\n',
 '外汇储备是一个国家持有的外币资产和黄金储备，用于平衡国际收支、稳定本国货币汇率和应对经济危机。\n',
 '杠杆率是指企业或投资者通过债务融资扩大资产规模的比例。高杠杆率可以放大收益，但也会增加风险。\n',
 '通货膨胀是指货币供应量增加导致物价普遍上升的现象。它反映了货币购买力的下降。\n',
 '有效市场假说认为，所有可用信息都已反映在资产价格中，资产价格总是处于合理水平。这意味着投资者难以通过分析公开信息获得超额收益。\n',
 '资产证券化是将缺乏流动性但有未来现金流的资产打包成证券出售的过程。它提高了资产的流动性，分散了风险。\n',
 '套期保值是一种通过金融衍生品对冲风险的操作。投资者通过在期货或期权市场建立相反的头寸，锁定价格波动风险。\n',
 '流动性风险是指资产无法在合理时间内以合理价格变现的风险。它通常与市场深度和资产的交易活跃度有关。\n',
 '资本充足率是银行自有资本与风险加权资产的比率，用于衡量银

In [20]:
from enum import Enum
class SearchType(str, Enum):
    """Enumerator of the types of search to perform."""

    similarity = "similarity"
    """Similarity search."""
    similarity_score_threshold = "similarity_score_threshold"
    """Similarity search with a score threshold."""
    mmr = "mmr"
    """Maximal Marginal Relevance reranking of similarity search."""

In [ ]:
def initial_vector_db(embedding_model=embeddings,question_list=question_list,answer_list=answer_list):
    # 创建小文本分割器,用于切割每个qa中的大文本块中的小文本
    small_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
    #将query文档进行通过Document类包装变成向量数据库的数据类型
    qa_list_document_list = [Document(page_content=q+"答案："+a) for q,a in zip(question_list,answer_list)]
    doc_ids = [str(uuid.uuid4()) for _ in qa_list_document_list] #为每一个文档创建唯一id
    sub_docs = []
    id_key = "doc_id"
    for i, doc in enumerate(qa_list_document_list):
        _id = doc_ids[i]  # 此时是父类的id---parent_id
        doc.metadata['type']="query_type" #将文档类型存储至元数据中
        _sub_doc = small_splitter.split_documents([doc]) #将父类文档拆分成若干个子类文档
        for _doc in _sub_doc:
            _doc.metadata[id_key]=_id #子文档的元数据存储父类文档的id
            sub_docs.append(_doc)
    store = InMemoryStore()
    # if os.path.exists("/home/wangshihang/projects/chat_API/chroma/qa_known_index"): 直接加载向量库,注意后面调通后需要修改
    # vectorstore2 = Chroma(persist_directory="/home/wangshihang/projects/chat_API/chroma/qa_index",
    #                       collection_name="qa_embedding",
    #                       embedding_function=embedding_model)
    # vectorstore_test = Chroma(persist_directory="/home/wangshihang/projects/chat_API/chroma/qa_known_online1", #qa_known_online
    #                       collection_name="qa_embedding",
    #                       embedding_function=embedding_model)
    #直接加载否则就是不断追加数据，加载数据
    vectorstore_test = Chroma(collection_name="qa_embedding",
                              collection_metadata={"hnsw:space": "cosine"},
                              embedding_function=embedding_model,
                             )  # embeddings  persist_directory="/home/wangshihang/projects/chat_API/chroma/qa_known_online1"
    qa_known_retriever = MultiVectorRetriever(vectorstore=vectorstore_test, docstore=store,
                                                   id_key=id_key,
                                                   search_kwargs={"k": 12,"score_threshold":0.5},
        search_type=SearchType.similarity_score_threshold)  # 确保使用相似度搜索) #先多拿一些子类片段,这样父类才能多链接一些出来，后面可以筛选
    qa_known_retriever.vectorstore.add_documents(sub_docs)
    qa_known_retriever.docstore.mset(list(zip(doc_ids, qa_list_document_list)))
    print("加载qa_known_retriever成功")
    return qa_known_retriever
qa_known_retriever=initial_vector_db() #实例化检索器

In [23]:
result=qa_known_retriever.invoke("什么是信用评级")
# result=qa_known_retriever.get_relevant_documents(query="专业评级机构（如穆迪、标普、惠誉）给出什么？")
result

[Document(metadata={'type': 'query_type'}, page_content='什么是信用利差？答案：信用利差是指不同信用等级债券收益率之间的差异。它反映了市场对信用风险的定价。\n'),
 Document(metadata={'type': 'query_type'}, page_content='解释什么是信用风险。答案：信用风险是指借款人或交易对手无法履行合同义务导致损失的风险。它通常通过信用评级和违约概率来衡量。\n'),
 Document(metadata={'type': 'query_type'}, page_content='解释什么是风险偏好。答案：风险偏好是指投资者愿意承担风险的程度，通常分为风险偏好型、风险中性型和风险厌恶型。\n'),
 Document(metadata={'type': 'query_type'}, page_content='解释什么是市场分割理论。答案：市场分割理论认为，不同期限的债券市场是相互独立的，投资者和发行人根据各自偏好选择特定市场。\n'),
 Document(metadata={'type': 'query_type'}, page_content='什么是利率期限结构？答案：利率期限结构是指不同期限债券收益率之间的关系，通常以收益率曲线表示。\n'),
 Document(metadata={'type': 'query_type'}, page_content='什么是零息债券？答案：零息债券是一种在发行时不支付利息，而是在到期时一次性支付本金和利息总额的债券。它通常以折价发行。\n')]